# 02 · The LMGEC baseline

**LMGEC** (Linear Multi-view Graph Embedding & Clustering) alternates three closed-form updates until convergence:

- **W** — per-view orthogonal projection (Procrustes / SVD)
- **G** — hard cluster assignment (nearest centroid)
- **F** — centroids (segment mean of the consensus embedding)

Views are fused with fixed a-priori weights **α**, derived from each view's initial inertia: `α_v ∝ exp(-inertia_v / T)`. Inertia is normalized by `√N` so the weights don't collapse to a single view on large datasets.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow

import numpy as np
import scipy.sparse as sp


def make_synthetic(n_per=60, k=3, dims=(30, 20), sep=1.0,
                   noise=(1.0, 2.5), p_noisy_edge=0.3, seed=0):
    """A small multi-view dataset with an uneven pair of views: (Xs, As, labels).

    Each view is Gaussian blobs around k centres. View 1 is deliberately
    noisier than view 0, and each graph mixes intra-cluster edges with a
    fraction of random ones -- so view weighting and propagation both have
    something real to do.
    """
    rng = np.random.RandomState(seed)
    n = n_per * k
    labels = np.repeat(np.arange(k), n_per)
    Xs, As = [], []
    for d, sigma in zip(dims, noise):
        centres = rng.randn(k, d) * sep
        Xs.append((centres[labels] + rng.randn(n, d) * sigma).astype(np.float64))
        A = np.zeros((n, n))
        for i in range(n):
            same = np.where(labels == labels[i])[0]
            A[i, rng.choice(same, size=5, replace=False)] = 1.0
            if rng.rand() < p_noisy_edge:
                A[i, rng.randint(n)] = 1.0          # a wrong edge
        As.append(sp.csr_matrix(np.maximum(A, A.T)))
    return Xs, As, labels


Xs, As, labels = make_synthetic()

# --- To use a real benchmark instead (downloads + caches on first call): ---
# from pymv.data import load_dataset
# data = load_dataset("acm")
# Xs, As, labels = data.Xs, data.As, data.labels

k = len(np.unique(labels))
print(f"views={len(Xs)}  samples={Xs[0].shape[0]}  clusters={k}")
print("view feature dims:", [X.shape[1] for X in Xs])

## Fit and inspect every output

In [ ]:
from pymv.models import LMGEC
from pymv.metrics import clustering_accuracy, nmi_score, ari_score

model = LMGEC(n_clusters=k, embedding_dim=16, temperature=1.0, max_iter=30)
model.fit(Xs, As)

print('ACC :', round(clustering_accuracy(labels, model.labels_), 4))
print('NMI :', round(nmi_score(labels, model.labels_), 4))
print('ARI :', round(ari_score(labels, model.labels_), 4))
print('view weights alphas :', np.round(model.alphas_, 3), '(sum=%.2f)' % model.alphas_.sum())
print('embedding shape     :', model.embedding_.shape)
print('centroids shape     :', model.centroids_.shape)
print('loss history length :', len(model.loss_history_))

## Convergence and the learned embedding

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(model.loss_history_, marker='o', ms=3)
ax1.set(title='Reconstruction loss', xlabel='iteration', ylabel='loss')
ax1.grid(alpha=.3)

Z2 = PCA(n_components=2).fit_transform(model.embedding_)
sc = ax2.scatter(Z2[:, 0], Z2[:, 1], c=model.labels_, cmap='tab10', s=18)
ax2.set(title='Consensus embedding (PCA)', xlabel='PC1', ylabel='PC2')
plt.tight_layout(); plt.show()

## The temperature knob

`temperature` controls how sharply α favours the lowest-inertia view. Low T → one view dominates; high T → uniform fusion. Watch the alphas move.

In [ ]:
for T in [0.1, 1.0, 10.0]:
    m = LMGEC(n_clusters=k, embedding_dim=16, temperature=T, max_iter=30).fit(Xs, As)
    acc = clustering_accuracy(labels, m.labels_)
    print(f'T={T:5.1f}  alphas={np.round(m.alphas_, 3)}  ACC={acc:.3f}')